In [24]:
!pip install -q git+https://github.com/RajdeepKushwaha5/QuashKV.git
!pip install -q transformers accelerate datasets sentencepiece protobuf

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [25]:
import torch, numpy as np, time, gc

print(f"PyTorch {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

PyTorch 2.10.0+cu128
CUDA: True
GPU: Tesla T4
VRAM: 14.6 GB


In [27]:
from quashkv import (
    LloydMaxCodebook, MSEQuantizer, InnerProductQuantizer,
    QuashKVEngine, QuashIndex, MixedPrecisionMSEQuantizer,
    pack_bits, unpack_bits,
)
from quashkv.codebook import compute_mse_cost
from quashkv.integrations.hf_cache import QuashKVCache
print("QuashKV imported successfully ✓")

QuashKV imported successfully ✓


In [29]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=DTYPE, device_map="auto", trust_remote_code=True,
)
model.eval()

config = model.config
n_layers = config.num_hidden_layers
n_kv_heads = getattr(config, "num_key_value_heads", config.num_attention_heads)
head_dim = config.hidden_size // config.num_attention_heads

print(f"Layers: {n_layers}, KV heads: {n_kv_heads}, Head dim: {head_dim}")
print(f"GPU mem used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Layers: 22, KV heads: 4, Head dim: 64
GPU mem used: 1.48 GB


In [30]:
import transformers
print(f"transformers {transformers.__version__}")

PROMPT = """The theory of general relativity, proposed by Albert Einstein in 1915, \
fundamentally changed our understanding of gravity. Rather than being a force acting \
at a distance, gravity is described as the curvature of spacetime caused by mass and \
energy. This elegant framework has been confirmed by numerous experiments, from the \
bending of light around massive objects to the detection of gravitational waves by \
LIGO in 2015. The implications extend far beyond physics, influencing our understanding \
of the universe's expansion, black holes, and the nature of time itself."""

inputs = tokenizer(PROMPT, return_tensors="pt").to(DEVICE)
seq_len = inputs["input_ids"].shape[1]
print(f"Prompt tokens: {seq_len}")

with torch.no_grad():
    outputs = model(**inputs, use_cache=True)
    raw_kv = outputs.past_key_values

# Extract KV cache — iterate and take first 2 tensors per layer (key, value)
past_kv = []
for layer_data in raw_kv:
    k, v = layer_data[0], layer_data[1]
    past_kv.append((k, v))
print(f"Extracted {len(past_kv)} layers via iteration")

k0, v0 = past_kv[0]
print(f"Layer 0 key shape:   {k0.shape}")
print(f"Layer 0 value shape: {v0.shape}")
print(f"Key dtype: {k0.dtype}")
print(f"Total KV cache: {sum(k.numel() + v.numel() for k, v in past_kv) * 2 / 1024**2:.1f} MB (fp16)")
print(f"Layers extracted: {len(past_kv)}")

transformers 5.0.0
Prompt tokens: 122
Extracted 22 layers via iteration
Layer 0 key shape:   torch.Size([1, 4, 122, 64])
Layer 0 value shape: torch.Size([1, 4, 122, 64])
Key dtype: torch.float16
Total KV cache: 2.6 MB (fp16)
Layers extracted: 22


In [31]:
print("=" * 65)
print(f"Compressing {MODEL_NAME} KV cache: keys=3-bit IP, values=3-bit MSE")
print("=" * 65)

all_results = []
for layer_idx, (keys, values) in enumerate(past_kv):
    keys_f = keys.float().cpu()
    values_f = values.float().cpu()
    B, H, S, D = keys_f.shape

    # Keys: InnerProductQuantizer returns (mse_indices, qjl_signs, norms)
    # Reconstruct via internal MSE quantizer for cosine/MSE comparison
    ip_q = InnerProductQuantizer(d=D, total_bits=3, seed=layer_idx)
    k_flat = keys_f.reshape(-1, D)
    k_norms = k_flat.norm(dim=-1, keepdim=True).clamp(min=1e-8)
    k_unit = k_flat / k_norms
    k_rot = ip_q.mse_quantizer.rotate(k_unit)
    k_mse_idx = ip_q.mse_quantizer.codebook.quantize_boundary(k_rot)
    k_rot_hat = ip_q.mse_quantizer.codebook.dequantize(k_mse_idx)
    k_hat = ip_q.mse_quantizer.unrotate(k_rot_hat) * k_norms
    k_hat = k_hat.reshape(B, H, S, D)

    # Values: MSEQuantizer returns (indices, norms)
    mse_q = MSEQuantizer(d=D, bits=3, seed=layer_idx + 1000)
    v_flat = values_f.reshape(-1, D)
    v_idx, v_norms = mse_q.compress(v_flat)
    v_hat = mse_q.decompress(v_idx, v_norms).reshape(B, H, S, D)

    k_mse = ((keys_f - k_hat) ** 2).mean().item()
    v_mse = ((values_f - v_hat) ** 2).mean().item()
    k_cos = torch.nn.functional.cosine_similarity(k_flat, k_hat.reshape(-1, D), dim=-1).mean().item()
    v_cos = torch.nn.functional.cosine_similarity(v_flat, v_hat.reshape(-1, D), dim=-1).mean().item()
    all_results.append({"layer": layer_idx, "k_mse": k_mse, "v_mse": v_mse, "k_cos": k_cos, "v_cos": v_cos})

print(f"\n{'Layer':>6} {'Key MSE':>10} {'Val MSE':>10} {'Key Cos':>10} {'Val Cos':>10}")
print("-" * 50)
for r in all_results:
    print(f"{r['layer']:>6} {r['k_mse']:>10.6f} {r['v_mse']:>10.6f} {r['k_cos']:>10.4f} {r['v_cos']:>10.4f}")

avg_k_cos = np.mean([r["k_cos"] for r in all_results])
avg_v_cos = np.mean([r["v_cos"] for r in all_results])
avg_k_mse = np.mean([r["k_mse"] for r in all_results])
avg_v_mse = np.mean([r["v_mse"] for r in all_results])
print(f"\n{'AVG':>6} {avg_k_mse:>10.6f} {avg_v_mse:>10.6f} {avg_k_cos:>10.4f} {avg_v_cos:>10.4f}")
print(f"\nTheoretical 3-bit MSE bound: {compute_mse_cost(head_dim, 3):.6f}")

Compressing TinyLlama/TinyLlama-1.1B-Chat-v1.0 KV cache: keys=3-bit IP, values=3-bit MSE

 Layer    Key MSE    Val MSE    Key Cos    Val Cos
--------------------------------------------------
     0   0.171609   0.000006     0.9437     0.9818
     1   0.561339   0.000029     0.9425     0.9838
     2   0.490541   0.000062     0.9409     0.9838
     3   0.529261   0.000453     0.9400     0.9838
     4   0.603159   0.000796     0.9430     0.9835
     5   0.645732   0.000813     0.9429     0.9833
     6   0.738702   0.000813     0.9414     0.9840
     7   0.528478   0.001035     0.9429     0.9836
     8   0.813295   0.001311     0.9433     0.9836
     9   0.773333   0.001160     0.9423     0.9838
    10   0.736123   0.001276     0.9402     0.9837
    11   0.692800   0.001827     0.9440     0.9835
    12   0.751049   0.001615     0.9410     0.9832
    13   0.737295   0.001722     0.9427     0.9838
    14   0.730708   0.001850     0.9432     0.9836
    15   0.755382   0.002562     0.9398    

In [32]:
print("Comparing attention: full precision vs 3-bit compressed\n")

attn_results = []
for layer_idx, (keys, values) in enumerate(past_kv):
    keys_f = keys.float().cpu()
    values_f = values.float().cpu()
    B, H, S, D = keys_f.shape

    engine = QuashKVEngine(head_dim=D, key_bits=3, value_bits=3, seed=layer_idx)
    engine.append(keys_f, values_f)

    queries = torch.randn(B, H, 1, D)
    scale = D ** -0.5
    scores_fp = (queries * scale) @ keys_f.transpose(-2, -1)
    weights_fp = torch.softmax(scores_fp, dim=-1)
    out_fp = weights_fp @ values_f

    out_compressed = engine.attention(queries)
    cos = torch.nn.functional.cosine_similarity(
        out_fp.reshape(-1, D), out_compressed.reshape(-1, D), dim=-1
    ).mean().item()
    attn_results.append(cos)

print(f"{'Layer':>6} {'Attn Cosine':>12}")
print("-" * 22)
for i, c in enumerate(attn_results):
    print(f"{i:>6} {c:>12.4f}")
avg_attn_cos = np.mean(attn_results)
print(f"\nAverage attention cosine: {avg_attn_cos:.4f}")

Comparing attention: full precision vs 3-bit compressed

 Layer  Attn Cosine
----------------------
     0       0.9945
     1       0.8792
     2       0.8967
     3       0.9350
     4       0.8498
     5       0.8512
     6       0.9151
     7       0.9297
     8       0.9568
     9       0.9176
    10       0.9292
    11       0.9175
    12       0.9099
    13       0.9664
    14       0.9503
    15       0.9624
    16       0.9230
    17       0.9401
    18       0.8879
    19       0.9372
    20       0.9392
    21       0.9336

Average attention cosine: 0.9237


In [33]:
print(f"{'Bits':>5} {'Key Cos':>10} {'Val Cos':>10} {'Key MSE':>10} {'Val MSE':>10} {'Ratio':>8}")
print("-" * 55)

for bits in [2, 3, 4]:
    res = []
    for li, (keys, values) in enumerate(past_kv):
        kf = keys.float().cpu(); vf = values.float().cpu()
        B, H, S, D = kf.shape

        # Keys via IP quantizer's internal MSE stage
        ip_q = InnerProductQuantizer(d=D, total_bits=bits, seed=li)
        k_flat = kf.reshape(-1, D)
        k_norms = k_flat.norm(dim=-1, keepdim=True).clamp(min=1e-8)
        k_unit = k_flat / k_norms
        k_rot = ip_q.mse_quantizer.rotate(k_unit)
        k_idx = ip_q.mse_quantizer.codebook.quantize_boundary(k_rot)
        k_rot_hat = ip_q.mse_quantizer.codebook.dequantize(k_idx)
        kh = (ip_q.mse_quantizer.unrotate(k_rot_hat) * k_norms).reshape(B, H, S, D)

        # Values via MSE quantizer
        mse_q = MSEQuantizer(d=D, bits=bits, seed=li+1000)
        vi, vn = mse_q.compress(vf.reshape(-1, D))
        vh = mse_q.decompress(vi, vn).reshape(B, H, S, D)

        res.append({
            "kc": torch.nn.functional.cosine_similarity(kf.reshape(-1,D), kh.reshape(-1,D), dim=-1).mean().item(),
            "vc": torch.nn.functional.cosine_similarity(vf.reshape(-1,D), vh.reshape(-1,D), dim=-1).mean().item(),
            "km": ((kf-kh)**2).mean().item(), "vm": ((vf-vh)**2).mean().item(),
        })
    ak = np.mean([r["kc"] for r in res]); av = np.mean([r["vc"] for r in res])
    mk = np.mean([r["km"] for r in res]); mv = np.mean([r["vm"] for r in res])
    print(f"{bits:>5} {ak:>10.4f} {av:>10.4f} {mk:>10.6f} {mv:>10.6f} {16/bits:>7.1f}x")

 Bits    Key Cos    Val Cos    Key MSE    Val MSE    Ratio
-------------------------------------------------------
    2     0.8011     0.9417   2.033955   0.007432     8.0x
    3     0.9420     0.9836   0.647757   0.002165     5.3x
    4     0.9837     0.9956   0.188484   0.000593     4.0x


In [34]:
total_orig = 0; total_comp = 0

for li, (keys, values) in enumerate(past_kv):
    engine = QuashKVEngine(head_dim=head_dim, key_bits=3, value_bits=3, seed=li, pack_storage=True)
    engine.append(keys.float().cpu(), values.float().cpu())
    total_orig += (keys.numel() + values.numel()) * 2
    total_comp += engine.actual_memory_bytes()

print(f"Original KV cache:   {total_orig / 1024:.1f} KB")
print(f"Compressed (packed): {total_comp / 1024:.1f} KB")
print(f"Compression ratio:   {total_orig / total_comp:.2f}x")
print(f"Memory saved:        {(1 - total_comp / total_orig) * 100:.1f}%")

Original KV cache:   2684.0 KB
Compressed (packed): 587.1 KB
Compression ratio:   4.57x
Memory saved:        78.1%


In [36]:
k0 = past_kv[0][0].float().cpu()
db_vectors = k0[0, 0, :, :]
print(f"Search DB: {db_vectors.shape[0]} vectors, dim={db_vectors.shape[1]}\n")

for bits in [2, 3, 4]:
    index = QuashIndex(dim=head_dim, bits=bits, seed=42)
    t0 = time.perf_counter()
    index.add(db_vectors)
    idx_time = (time.perf_counter() - t0) * 1000
    recall = index.recall_at_k(db_vectors[:10], db_vectors, k=5)
    print(f"{bits}-bit: Recall@5={recall:.3f}, Ratio={index.compression_ratio():.1f}x, Index={idx_time:.1f}ms")

Search DB: 122 vectors, dim=64

2-bit: Recall@5=0.560, Ratio=1.9x, Index=0.6ms
3-bit: Recall@5=0.680, Ratio=1.9x, Index=0.5ms
4-bit: Recall@5=0.800, Ratio=1.9x, Index=0.6ms


In [37]:
print("=" * 60)
print(f"QuashKV Validation Report — {MODEL_NAME}")
print("=" * 60)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Layers: {n_layers}, KV heads: {n_kv_heads}, Head dim: {head_dim}")
print(f"\n--- 3-bit Compression ---")
print(f"Avg key cosine:     {avg_k_cos:.4f}")
print(f"Avg value cosine:   {avg_v_cos:.4f}")
print(f"Avg attention cos:  {avg_attn_cos:.4f}")
print(f"Compression ratio:  {total_orig / total_comp:.2f}x")
print(f"Memory saved:       {(1 - total_comp / total_orig) * 100:.1f}%")
print(f"\n--- Theoretical ---")
print(f"MSE bound (3-bit):  {compute_mse_cost(head_dim, 3):.6f}")
print(f"Empirical key MSE:  {avg_k_mse:.6f}")
print(f"Empirical val MSE:  {avg_v_mse:.6f}")
print(f"\n✅ QuashKV validated on real {MODEL_NAME} KV caches!")

QuashKV Validation Report — TinyLlama/TinyLlama-1.1B-Chat-v1.0
GPU: Tesla T4
Layers: 22, KV heads: 4, Head dim: 64

--- 3-bit Compression ---
Avg key cosine:     0.9420
Avg value cosine:   0.9836
Avg attention cos:  0.9237
Compression ratio:  4.57x
Memory saved:       78.1%

--- Theoretical ---
MSE bound (3-bit):  0.034548
Empirical key MSE:  0.647757
Empirical val MSE:  0.002165

✅ QuashKV validated on real TinyLlama/TinyLlama-1.1B-Chat-v1.0 KV caches!


In [38]:
del model, past_kv
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed.")


GPU memory freed.


In [40]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, numpy as np, time, gc

DEVICE = "cuda"
DTYPE = torch.float16
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=DTYPE, device_map="auto", trust_remote_code=True,
)
model.eval()
config = model.config
n_layers = config.num_hidden_layers
n_kv_heads = getattr(config, "num_key_value_heads", config.num_attention_heads)
head_dim = config.hidden_size // config.num_attention_heads

from quashkv import MSEQuantizer, InnerProductQuantizer, QuashKVEngine
from quashkv.codebook import compute_mse_cost
print(f"Model loaded: {n_layers} layers, {n_kv_heads} KV heads, head_dim={head_dim}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded: 22 layers, 4 KV heads, head_dim=64


In [41]:
seed_prompt = "The history of artificial intelligence began in the 1950s when"
inputs = tokenizer(seed_prompt, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    generated = model.generate(
        **inputs, max_new_tokens=1024, do_sample=False,
        use_cache=True, return_dict_in_generate=True,
    )

total_tokens = generated.sequences.shape[1]
print(f"Total tokens: {total_tokens}")

# Now do a forward pass on the full generated sequence to get the KV cache
with torch.no_grad():
    outputs = model(generated.sequences, use_cache=True)
    raw_kv = outputs.past_key_values

past_kv = []
for layer_data in raw_kv:
    k, v = layer_data[0], layer_data[1]
    past_kv.append((k, v))

k0, v0 = past_kv[0]
print(f"KV shape: {k0.shape}  (expect ~1K+ seq_len)")
print(f"Total KV cache: {sum(k.numel() + v.numel() for k, v in past_kv) * 2 / 1024**2:.1f} MB (fp16)")


Total tokens: 1040
KV shape: torch.Size([1, 4, 1040, 64])  (expect ~1K+ seq_len)
Total KV cache: 22.3 MB (fp16)


In [42]:
all_results = []
for layer_idx, (keys, values) in enumerate(past_kv):
    keys_f = keys.float().cpu()
    values_f = values.float().cpu()
    B, H, S, D = keys_f.shape

    ip_q = InnerProductQuantizer(d=D, total_bits=3, seed=layer_idx)
    k_flat = keys_f.reshape(-1, D)
    k_norms = k_flat.norm(dim=-1, keepdim=True).clamp(min=1e-8)
    k_unit = k_flat / k_norms
    k_rot = ip_q.mse_quantizer.rotate(k_unit)
    k_idx = ip_q.mse_quantizer.codebook.quantize_boundary(k_rot)
    k_rot_hat = ip_q.mse_quantizer.codebook.dequantize(k_idx)
    k_hat = (ip_q.mse_quantizer.unrotate(k_rot_hat) * k_norms).reshape(B, H, S, D)

    mse_q = MSEQuantizer(d=D, bits=3, seed=layer_idx + 1000)
    v_flat = values_f.reshape(-1, D)
    v_idx, v_norms = mse_q.compress(v_flat)
    v_hat = mse_q.decompress(v_idx, v_norms).reshape(B, H, S, D)

    k_cos = torch.nn.functional.cosine_similarity(k_flat, k_hat.reshape(-1, D), dim=-1).mean().item()
    v_cos = torch.nn.functional.cosine_similarity(v_flat, v_hat.reshape(-1, D), dim=-1).mean().item()
    all_results.append({"k_cos": k_cos, "v_cos": v_cos})

avg_k = np.mean([r["k_cos"] for r in all_results])
avg_v = np.mean([r["v_cos"] for r in all_results])
print(f"Long sequence ({k0.shape[2]} tokens) 3-bit quality:")
print(f"  Key cosine:   {avg_k:.4f}")
print(f"  Value cosine: {avg_v:.4f}")



Long sequence (1040 tokens) 3-bit quality:
  Key cosine:   0.9420
  Value cosine: 0.9835


In [43]:
total_orig = 0; total_comp = 0
for li, (keys, values) in enumerate(past_kv):
    engine = QuashKVEngine(head_dim=head_dim, key_bits=3, value_bits=3, seed=li, pack_storage=True)
    engine.append(keys.float().cpu(), values.float().cpu())
    total_orig += (keys.numel() + values.numel()) * 2
    total_comp += engine.actual_memory_bytes()

seq_len = past_kv[0][0].shape[2]
print(f"\nLong-Sequence Results ({seq_len} tokens, TinyLlama 1.1B)")
print(f"  Original:    {total_orig / 1024:.1f} KB")
print(f"  Compressed:  {total_comp / 1024:.1f} KB")
print(f"  Ratio:       {total_orig / total_comp:.2f}x")
print(f"  Saved:       {(1 - total_comp / total_orig) * 100:.1f}%")
print(f"  Key cosine:  {avg_k:.4f}")
print(f"  Val cosine:  {avg_v:.4f}")
print(f"\nComparison: 122 tokens -> {seq_len} tokens = quality stable? {'YES' if avg_k > 0.93 else 'CHECK'}")


Long-Sequence Results (1040 tokens, TinyLlama 1.1B)
  Original:    22880.0 KB
  Compressed:  5005.0 KB
  Ratio:       4.57x
  Saved:       78.1%
  Key cosine:  0.9420
  Val cosine:  0.9835

Comparison: 122 tokens -> 1040 tokens = quality stable? YES


In [44]:
del model, past_kv
gc.collect(); torch.cuda.empty_cache()
print("Done.")

Done.
